# GONet CDR baseline trên Kaggle

Notebook này chạy segmenter đã có checkpoint để tạo OD/OC masks, tính vertical CDR, rồi tính AUROC. Nó **không train classifier**. RDR hiện chưa có công thức mask-level đã xác nhận nên được để trống.

Yêu cầu: bật Internet và GPU; gắn inputs `DRISHTI-GS`, `HYDR-dataset`, `Papila` và `fundu-segmenter-weights`. Không cần REFUGE/REFUGE2 vì hiện không có nhãn glaucoma tương ứng. Code baseline phải được push lên GitHub `main` trước khi chạy cell đồng bộ repo.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/TaiTranDang145/Glaucoma-Detection.git"
REPO_DIR = Path("/kaggle/working/Glaucoma-Detection")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout; remove or rename it, then rerun.")
else:
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

required_files = [
    REPO_DIR / "src/models/fundu_segmenter_adapter.py",
    REPO_DIR / "src/infer_disc_baselines.py",
    REPO_DIR / "src/evaluate_disc_baselines.py",
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise RuntimeError("Baseline code is not on GitHub main yet. Push the local implementation, then rerun this cell. Missing: " + ", ".join(missing))
os.chdir(REPO_DIR)
print(f"Repository ready: {REPO_DIR}")

In [ ]:
# Keep Kaggle's installed PyTorch/CUDA. Install only the extra Python packages used by the adapter and scripts.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "timm==1.0.20", "einops==0.8.1", "ml-collections==1.1.0",
    "omegaconf", "scikit-learn", "matplotlib",
], check=True)
print("Dependencies installed.")

In [ ]:
# Clone the authors' model implementation; the checkpoint alone does not contain Python architecture code.
FUNDUS_REPO = Path("/kaggle/working/FunduSegmenter")
FUNDUS_URL = "https://github.com/JusticeZzy/FunduSegmenter.git"
if (FUNDUS_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(FUNDUS_REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", FUNDUS_URL, str(FUNDUS_REPO)], check=True)
os.environ["FUNDUS_SEGMENTER_REPO"] = str(FUNDUS_REPO)
print(f"FunduSegmenter source: {FUNDUS_REPO}")

In [ ]:
# These mount paths match the checkpoint path shown in this Kaggle notebook.
KAGGLE_OWNER_ROOT = Path("/kaggle/input/datasets/taitrandang")
if not KAGGLE_OWNER_ROOT.is_dir():
    raise FileNotFoundError(f"Kaggle input root not found: {KAGGLE_OWNER_ROOT}; inspect /kaggle/input and update this path.")
mounts = [path for path in KAGGLE_OWNER_ROOT.iterdir() if path.is_dir()]
print("Attached dataset folders:", [path.name for path in mounts])

def find_mount(token):
    found = [path for path in mounts if token in path.name.casefold()]
    if len(found) != 1:
        raise RuntimeError(f"Expected one attached dataset matching {token!r}; found: {found}")
    return found[0]

def replace_link(alias, target):
    link = DATA_ROOT / alias
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise FileExistsError(f"Refusing to replace existing non-link path: {link}")
    link.symlink_to(target, target_is_directory=True)

DATA_ROOT = Path("/kaggle/working/gonet-data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
hydr_mount = find_mount("hydr")
hydr_roots = [p.parent for p in hydr_mount.rglob("Labels.csv") if (p.parent / "Images").is_dir()]
if len(hydr_roots) != 1:
    raise RuntimeError(f"HYDR needs one Labels.csv with a sibling Images/ directory; found: {hydr_roots}")
replace_link("HYDR", hydr_roots[0])
replace_link("DRISHTI-GS", find_mount("drishti"))
replace_link("PAPILA", find_mount("papila"))

papila_root = DATA_ROOT / "PAPILA"
if not any(p.is_dir() for p in papila_root.rglob("FundusImages")) or not any(p.is_dir() for p in papila_root.rglob("ClinicalData")):
    raise RuntimeError("PAPILA input must contain FundusImages/ and ClinicalData/. Check the attached dataset layout.")
CHECKPOINT = find_mount("fundu-segmenter-weights") / "FunduSegmenter_OriginalImage.pth"
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}")
print("Data root:", DATA_ROOT)
print("Checkpoint:", CHECKPOINT, f"({CHECKPOINT.stat().st_size / 1024**3:.2f} GiB)")

## Tạo manifest và kiểm tra nhãn

Manifest builder bỏ qua dataset folder không có mặt. Chỉ các bộ có nhãn glaucoma được đưa vào AUROC; không cần gắn REFUGE2. Kiểm tra số ảnh mỗi domain trước khi chạy segmenter.

In [ ]:
import pandas as pd

MANIFEST = Path("/kaggle/working/gonet_msd.csv")
subprocess.run([
    sys.executable, "-m", "src.prepare_data",
    "--data-root", str(DATA_ROOT), "--output", str(MANIFEST),
], cwd=REPO_DIR, check=True)
manifest = pd.read_csv(MANIFEST)
display(manifest.groupby(["domain", "label"]).size().unstack(fill_value=0))
print(f"Total labeled images: {len(manifest)}")
if manifest.empty:
    raise RuntimeError("Manifest is empty; check the input labels and folder aliases.")

## Chạy inference baseline

FunduSegmenter tạo OD/OC masks; pipeline lưu CDR từng ảnh. Tác giả FunduSegmenter cho biết checkpoint đã thấy Drishti-GS và REFUGE train/validation trong lúc train segmentation, nên không xem AUROC hai domain đó là đánh giá độc lập.

In [ ]:
import torch
from omegaconf import OmegaConf

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle Notebook Settings before running segmentation inference.")

OUTPUT_DIR = Path("/kaggle/working/disc-baselines")
BASELINE_CONFIG = Path("/kaggle/working/disc_baselines.yaml")
cfg = OmegaConf.load(REPO_DIR / "configs/disc_baselines.yaml")
cfg.paths.data_root = str(DATA_ROOT)
cfg.paths.manifest = str(MANIFEST)
cfg.paths.output_dir = str(OUTPUT_DIR)
cfg.segmenter.checkpoint = str(CHECKPOINT)
cfg.segmenter.device = "cuda"
OmegaConf.save(cfg, BASELINE_CONFIG)

subprocess.run([
    sys.executable, "-m", "src.infer_disc_baselines",
    "--config", str(BASELINE_CONFIG),
], cwd=REPO_DIR, check=True)
RESULTS = OUTPUT_DIR / "disc_baselines.csv"
print("Inference results:", RESULTS)

## Tính AUROC và xem kết quả

CDR được dùng trực tiếp làm score GON+. RDR chưa được đưa vào ROC vì định nghĩa đo từ mask chưa xác nhận.

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.evaluate_disc_baselines",
    "--results", str(RESULTS), "--output-dir", str(OUTPUT_DIR),
], cwd=REPO_DIR, check=True)
auc_path = OUTPUT_DIR / "auc_comparison.csv"
display(pd.read_csv(auc_path))
print("Outputs saved under:", OUTPUT_DIR)

In [ ]:
# Optional: export overlays for a few fundus samples.
subprocess.run([
    sys.executable, "-m", "src.visualize_disc_baselines",
    "--results", str(RESULTS),
    "--data-root", str(DATA_ROOT),
    "--results-root", str(OUTPUT_DIR),
    "--output-dir", str(OUTPUT_DIR / "visualizations"),
    "--count", "8", "--seed", "42",
], cwd=REPO_DIR, check=True)
print("Overlays:", OUTPUT_DIR / "visualizations")